In [53]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import json
import requests
from io import BytesIO
import os
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, recall_score, precision_score
from torch.optim import AdamW
from tqdm import tqdm
from datasets import load_metric

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


###Modules for transforming and loading data

In [56]:
class CustomImageDataset(Dataset):
    def __init__(self, json_file, img_dir, transform=None):
        with open(json_file, 'r') as f:
            self.data = json.load(f)
        self.img_dir = img_dir
        self.transform = transform
        self.label2idx = {'Non-sarcasm': 0, 'Sarcasm': 1}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = os.path.join(self.img_dir, item['image'])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image_tensor = self.transform(image)
        else:
            image_tensor = transforms.ToTensor()(image)

        label = self.label2idx[item['label']]
        return {'pixel_values': image_tensor, 'labels': label}

In [57]:
preprocess_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

###Data Loader

for SILVER

In [58]:
base_img_dir = "/content/drive/MyDrive/Colab/Multimodal/Data/3k_image"
img_json_gold = "/content/drive/MyDrive/Colab/Multimodal/Data/GOLD/Image modality"
img_json = "/content/drive/MyDrive/Colab/Multimodal/Data/GOLD"
train_json = os.path.join(img_json_gold, "train_gold.json")
val_json   = os.path.join(img_json_gold, "dev_gold.json")
test_json  = os.path.join(img_json_gold, "test_gold.json")
ckpt_dir   = "/content/drive/MyDrive/Colab/Multimodal/Image/Pretrained_ResNet/gold"
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path  = os.path.join(ckpt_dir, "best_resnet_model_gold.pth")

for SILVER+GOLD

In [59]:
dataset_train = CustomImageDataset(train_json, base_img_dir, transform=preprocess_transform)
dataset_val = CustomImageDataset(val_json, base_img_dir, transform=preprocess_transform)
dataset_test = CustomImageDataset(test_json, base_img_dir, transform=preprocess_transform)

In [60]:
def collate_fn(batch):
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    labels = torch.tensor([item['labels'] for item in batch])
    return {'pixel_values': pixel_values, 'labels': labels}

In [61]:
batch_size = 128
train_loader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(dataset_test, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

###Model

In [62]:
model = models.resnet50(pretrained=True)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [63]:
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

###Training settings

In [85]:
learning_rate = 2e-4
num_epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

####Evaluate Function

In [86]:
def compute_metrics(predictions, references):
    predictions = np.array(predictions)
    references = np.array(references)

    accuracy = accuracy_score(references, predictions)
    f1 = f1_score(references, predictions, average='binary', pos_label=1)
    precision = precision_score(references, predictions, average='binary', pos_label=1)
    recall = recall_score(references, predictions, average='binary', pos_label=1)

    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }

###Training

In [87]:
best_f1 = -1.0
patience = 5
counter = 0
save_dir = "/content/drive/MyDrive/Colab/Multimodal/Image/Pretrained_ResNet/gold"

for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        inputs = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    val_metrics = compute_metrics(all_preds, all_labels)
    f1 = val_metrics['f1']
    print(f"Epoch {epoch+1} — F1 on dev: {f1:.4f}")

    # Early Stopping check
    if f1 > best_f1:
        best_f1 = f1
        counter = 0
        torch.save(model.state_dict(), os.path.join(save_dir, "best_resnet_model.pth"))
        print("New best model saved.")
    else:
        counter += 1
        print(f"No improvement. Patience: {counter}/{patience}")
        if counter >= patience:
            print("Early stopping triggered.")
            break

Epoch 1 — F1 on dev: 0.0364
New best model saved.
Epoch 2 — F1 on dev: 0.0741
New best model saved.
Epoch 3 — F1 on dev: 0.4034
New best model saved.
Epoch 4 — F1 on dev: 0.4328
New best model saved.
Epoch 5 — F1 on dev: 0.3871
No improvement. Patience: 1/5
Epoch 6 — F1 on dev: 0.4231
No improvement. Patience: 2/5
Epoch 7 — F1 on dev: 0.4219
No improvement. Patience: 3/5
Epoch 8 — F1 on dev: 0.3883
No improvement. Patience: 4/5
Epoch 9 — F1 on dev: 0.3596
No improvement. Patience: 5/5
Early stopping triggered.


###Evaluate

In [90]:
ckpt_path = "/content/drive/MyDrive/Colab/Multimodal/Image/Pretrained_ResNet/gold/best_resnet_model.pth"
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        inputs = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

test_metrics = compute_metrics(all_preds, all_labels)

Testing: 100%|██████████| 2/2 [00:02<00:00,  1.39s/it]


In [91]:
test_metrics

{'accuracy': 0.655,
 'f1': 0.43902439024390244,
 'precision': 0.375,
 'recall': 0.5294117647058824}